# 04 - Event Detection

Detect interesting driving events (hard braking, sudden acceleration, stops, turns).

## Steps:
1. Load cleaned trajectory data
2. Detect events
3. Summarize events per trip
4. Save event markers


In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent / "src"))

import pandas as pd

from src.events import detect_all_events, get_event_summary

# Load cleaned data
data_dir = Path("../data/processed")
df = pd.read_parquet(data_dir / "trajectories_cleaned.parquet")

print(f"Loaded {len(df)} GPS points from {df['trajectory_id'].nunique()} trajectories")


## Detect Events

Find hard brakes, sudden accelerations, stops, and sharp turns.


In [ ]:
# Detect all events
df_with_events = detect_all_events(
    df,
    brake_threshold=-3.0,
    accel_threshold=3.0,
    stop_threshold=1.0,
    turn_threshold=30.0
)

print("\nEvent detection complete")
df_with_events.head()


## Event Summary

Count events per trip.


In [ ]:
# Get event summary
event_summary = get_event_summary(df_with_events)

print("\nEvent summary per trip:")
print(event_summary.describe())

# Show trips with most events
print("\nTop 10 trips by total events:")
event_summary["total_events"] = (
    event_summary["n_hard_brakes"] +
    event_summary["n_sudden_accels"] +
    event_summary["n_sharp_turns"]
)
print(event_summary.nlargest(10, "total_events")[['trajectory_id', 'n_hard_brakes', 'n_sudden_accels', 'n_sharp_turns', 'total_events']])


## Save Results

Save trajectory data with event markers.


In [ ]:
# Save data with events
output_file = data_dir / "trajectories_with_events.parquet"
df_with_events.to_parquet(output_file, index=False)
print(f"Saved data with events to {output_file}")

# Save event summary
summary_file = data_dir / "event_summary.csv"
event_summary.to_csv(summary_file, index=False)
print(f"Saved event summary to {summary_file}")
